# KDD Process Volcano Data Analysis

In [72]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from prophet import Prophet
import plotly.graph_objects as go 
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"
from dash import Dash, dcc, html, Input, Output

## Data Cleaning and Preprocessing

In [73]:
def load_data(filepath="volcano-events.tsv"):
    try:
        df = pd.read_csv(filepath, sep='\t')
    except FileNotFoundError:
        print(f"Error: File '{filepath}' not found.")
        return pd.DataFrame()

    df.rename(columns={
        'Damage ($Mil)': 'Damage_Millions',
        'Total Damage ($Mil)': 'Total_Damage_Millions',
        'Elevation (m)': 'Elevation',
        'Total Deaths': 'Total_Deaths',
        'Total Injuries': 'Total_Injuries'
    }, inplace=True)

    df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
    df.dropna(subset=['Year'], inplace=True)
    df['VEI'] = pd.to_numeric(df['VEI'], errors='coerce')
    df['Deaths'] = pd.to_numeric(df['Deaths'], errors='coerce').fillna(0)
    df['Damage_Millions'] = pd.to_numeric(df['Damage_Millions'], errors='coerce').fillna(0)
    df.dropna(subset=['Latitude', 'Longitude', 'Country'], inplace=True)
    return df

In [74]:
# Load data for notebook analysis (outside of the dashboard)
df = load_data()
df.head()

,Search Parameters,Year,Mo,Dy,Tsu,Eq,Name,Location,Country,Latitude,...,Total_Deaths,Total Death Description,Total Missing,Total Missing Description,Total_Injuries,Total Injuries Description,Total_Damage_Millions,Total Damage Description,Total Houses Destroyed,Total Houses Destroyed Description
1,NaN,-4360.0,NaN,NaN,NaN,NaN,Macauley,Kermadec Is,New Zealand,-30.210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,-4350.0,NaN,NaN,NaN,NaN,Kikai,Ryukyu Is,Japan,30.793,...,NaN,3.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
3,NaN,-4050.0,NaN,NaN,NaN,NaN,Masaya,Nicaragua,Nicaragua,11.985,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,-4000.0,NaN,NaN,NaN,NaN,Witori,New Britain-SW Pac,Papua New Guinea,-5.576,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN
5,NaN,-3580.0,NaN,NaN,NaN,NaN,Taal,Luzon-Philippines,Philippines,14.002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Spatial Analysis

In [75]:
def get_map_figure(df):
    df_map = df.copy()
    df_map['VEI_Size'] = df_map['VEI'].fillna(0.5)

    fig = px.scatter_geo(
        df_map,
        lat="Latitude",
        lon="Longitude",
        color="Type",
        size="VEI_Size",
        hover_name="Name",
        hover_data={"Country": True, "Year": True, "Deaths": True, "VEI_Size": False},
        title="Global Volcano Distribution",
        projection="natural earth",
        size_max=15,
        template="plotly_dark",
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    fig.update_layout(
        margin={"r":0,"t":50,"l":0,"b":0},
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Temporal Analysis of volcanic Eruptions

In [76]:
# ============================
# 1. Yearly aggregation helper
# ============================

def build_eruptions_per_year(df):
    """
    Build a yearly time series: index = Year, value = number of eruptions.
    We drop missing or invalid (<= 0) years.
    """
    temp = df.dropna(subset=["Year"]).copy()
    temp = temp[temp["Year"] > 0]

    eruptions_per_year = (
        temp
        .groupby("Year")
        .size()
        .sort_index()
    )

    eruptions_per_year.index = eruptions_per_year.index.astype(int)
    eruptions_per_year.name = "eruptions_per_year"

    return eruptions_per_year


In [77]:
# =====================================
# 2. Aggregation by year or by century
# =====================================

def aggregate_eruptions(df, by="year"):
    """
    Aggregate eruptions by year or by century.
    Returns a dataframe with two columns: period, count.
    """
    temp = df.dropna(subset=["Year"]).copy()
    temp = temp[temp["Year"] > 0]  # keep AD years only

    if by == "year":
        grouped = (
            temp.groupby("Year")
            .size()
            .reset_index(name="count")
        )
        grouped.rename(columns={"Year": "period"}, inplace=True)

    elif by == "century":
        # Century numbering: 1..n (e.g. 19 = 1801-1900)
        temp["century"] = ((temp["Year"] - 1) // 100 + 1).astype(int)
        grouped = (
            temp.groupby("century")
            .size()
            .reset_index(name="count")
        )
        grouped.rename(columns={"century": "period"}, inplace=True)

    else:
        raise ValueError("by must be 'year' or 'century'")

    return grouped

In [78]:
# ====================================
# 3. Plotly time series (year/century)
# ====================================

def render_time_series(df, by="year"):
    """
    Build a time-series figure of eruption frequency over time
    using Plotly Express, either per year or per century.
    """
    agg = aggregate_eruptions(df, by=by)

    if by == "year":
        x_label = "Year"
        title = "Number of eruptions per year"
    else:
        x_label = "Century"
        title = "Number of eruptions per century"

    fig = px.line(
        agg,
        x="period",
        y="count",
        markers=True,
        labels={"period": x_label, "count": "Number of eruptions"},
        title=title,
        color_discrete_sequence=['#ff5722']  # orange/red curve
    )

    fig.update_layout(
        height=320,
        paper_bgcolor='rgba(0,0,0,0)',   # transparent to fit the dark card
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white"),
        xaxis=dict(
            tickmode="auto",
            tickangle=-45
        ),
        margin=dict(l=50, r=20, t=60, b=40)
    )

    fig.update_traces(
        hovertemplate=f"{x_label}: %{{x}}<br>Eruptions: %{{y}}<extra></extra>"
    )

    return fig

In [79]:
# =====================================================
# 4. Time series mode switcher for the dashboard (year /
#    century / smoothed yearly series with moving average)
# =====================================================

def render_temporal_series(df, mode="year"):
    """
    3 modes for the dashboard:
    - 'year'    : time series per year (uses render_time_series)
    - 'century' : time series per century (uses render_time_series)
    - 'smooth'  : yearly series + 10-year moving average (Plotly)
    """
    # 1) Yearly series (raw)
    if mode == "year":
        return render_time_series(df, by="year")

    # 2) Century series (raw)
    if mode == "century":
        return render_time_series(df, by="century")

    # 3) Smoothed yearly series (10-year moving average)
    if mode == "smooth":
        per_year = build_eruptions_per_year(df)
        smooth = per_year.rolling(window=10).mean()

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=per_year.index,
            y=per_year.values,
            mode="lines",
            name="Raw yearly data",
            line=dict(width=1, color="#888")   # grey, less dominant
        ))
        fig.add_trace(go.Scatter(
            x=smooth.index,
            y=smooth.values,
            mode="lines",
            name="10-year moving average",
            line=dict(width=3, color="#ff5722")  # main orange/red curve
        ))

        fig.update_layout(
            title="Smoothed eruptions per year (10-year moving average)",
            xaxis_title="Year",
            yaxis_title="Number of eruptions",
            height=320,
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
            font=dict(color="white"),
            margin=dict(l=50, r=20, t=60, b=40)
        )

        fig.update_traces(
            hovertemplate="Year: %{x}<br>Eruptions: %{y}<extra></extra>"
        )

        return fig

    # Fallback (safety)
    return render_time_series(df, by="year")

In [80]:
# =========================
# 5. Prophet data preparation
# =========================

def prepare_prophet_df(eruptions_per_year, min_year=1800):
    """
    Convert the yearly eruption series into Prophet's expected format.
    Keep only modern years (>= min_year) for a cleaner and valid forecast.
    """
    # Series -> DataFrame
    df_prophet = eruptions_per_year.reset_index()
    df_prophet.columns = ["Year", "y"]

    # Keep only modern years (for datetime + data quality reasons)
    df_prophet = df_prophet[df_prophet["Year"] >= min_year]

    # Create a datetime column 'ds' (required by Prophet)
    df_prophet["ds"] = pd.to_datetime(
        df_prophet["Year"].astype(int).astype(str) + "-01-01"
    )

    return df_prophet[["ds", "y"]]

Restricting the time range for forecasting

The original volcanic eruption dataset spans several millennia, including
very early years (e.g. 200 AD). However, pandas datetime objects and the
Prophet model are not designed to handle such ancient dates directly.

To build a meaningful and technically valid forecast, we restrict the
time range to **modern years** (e.g. from 1800 onwards). This also makes
sense from a data quality perspective: recent centuries have much more
complete and reliable recording of volcanic activity.

The forecasting model is therefore fitted only on this modern subset of
the time series, and the predictions should be interpreted within this
context.


In [81]:
# =====================
# 6. Prophet forecasting
# =====================

def forecast_eruptions(df_prophet, n_future_years=20):
    """
    Fit a Prophet model on the historical data and forecast n_future_years ahead.
    """
    model = Prophet()
    model.fit(df_prophet)

    # Generate future dates (one point per year)
    future_dates = model.make_future_dataframe(periods=n_future_years, freq="YE")

    # Predict on both historical + future dates
    prediction = model.predict(future_dates)

    return prediction

In [82]:
# ==================================
# 7. Plotly figure for the forecast
# ==================================

def make_forecast_figure(df_prophet, prediction):
    """
    Build a Plotly figure showing historical yearly eruptions
    and Prophet forecast.
    """
    fig = go.Figure()

    # Historical data
    fig.add_trace(go.Scatter(
        x=df_prophet["ds"],
        y=df_prophet["y"],
        mode="lines",
        name="Historical",
        line=dict(color="#ff5722", width=2)
    ))

    # Forecast
    fig.add_trace(go.Scatter(
        x=prediction["ds"],
        y=prediction["yhat"],
        mode="lines",
        name="Forecast",
        line=dict(color="#ff9800", width=2, dash="dash")  # orange, dashed
    ))

    fig.update_layout(
        title="Forecast of volcanic eruptions per year (Prophet)",
        xaxis_title="Year",
        yaxis_title="Number of eruptions",
        height=320,
        paper_bgcolor='rgba(0,0,0,0)',   # same as other cards
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white"),
        margin=dict(l=50, r=20, t=60, b=40),
        legend=dict(bgcolor="rgba(0,0,0,0)")
    )

    return fig


In [83]:
def make_empty_forecast_figure(message):
    """
    Simple empty figure used when there is not enough data
    to fit a reliable Prophet model (e.g. after filters).
    """
    fig = go.Figure()
    fig.update_layout(
        title=message,
        xaxis_title="Time",
        yaxis_title="Number of eruptions",
        height=320,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white"),
        margin=dict(l=50, r=20, t=60, b=40)
    )
    return fig

In [84]:
# Yearly time series (raw)
fig_ts_year = render_time_series(df, by="year")
fig_ts_year.show()

# Time series with moving average (smooth mode)
fig_ts_smooth = render_temporal_series(df, mode="smooth")
fig_ts_smooth.show()

# Forecast
eruptions_per_year = build_eruptions_per_year(df)
df_prophet = prepare_prophet_df(eruptions_per_year, min_year=1800)
prediction = forecast_eruptions(df_prophet, n_future_years=50)
fig_forecast = make_forecast_figure(df_prophet, prediction)
fig_forecast.show()

/usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found
/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not found
/usr/bin/xdg-open: 882: epiphany: not found
/usr/bin/xdg-open: 882: konqueror: not found
/usr/bin/xdg-open: 882: chromium: not found
/usr/bin/xdg-open: 882: chromium-browser: not found
/usr/bin/xdg-open: 882: google-chrome: not found
/usr/bin/xdg-open: 882: www-browser: not found
/usr/bin/xdg-open: 882: links2: not found
/usr/bin/xdg-open: 882: elinks: not found
/usr/bin/xdg-open: 882: links: not found
/usr/bin/xdg-open: 882: lynx: not found
/usr/bin/xdg-open: 882: w3m: not found
xdg-open: no method available for opening 'http://127.0.0.1:40361'
/usr/bin/xdg-open: 882: x-www-browser: not found
/usr/bin/xdg-open: 882: firefox: not found
/usr/bin/xdg-open: 882: iceweasel: not found
/usr/bin/xdg-open: 882: seamonkey: not found
/usr/bin/xdg-open: 882: mozilla: not foun

## Impact Analysis

In [85]:
def get_impact_figure(df):
    top_deadly = df.nlargest(10, 'Deaths').sort_values('Deaths', ascending=True)
    fig = px.bar(
        top_deadly,
        x="Deaths",
        y="Name",
        orientation='h',
        text="Deaths",
        title="Top 10 Deadliest Eruptions",
        template="plotly_dark"
    )
    fig.update_traces(marker_color='#ff5722', textposition='outside')
    fig.update_layout(
        xaxis_title="Deaths", 
        yaxis_title="",
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## Correlation Analysis

In [86]:
def get_correlation_figure(df):
    damage_df = df[df['Damage_Millions'] > 0].copy()
    if damage_df.empty:
         return px.scatter(title="No Data")

    fig = px.scatter(
        damage_df,
        x="VEI",
        y="Damage_Millions",
        size="Deaths",
        hover_name="Name",
        log_y=True,
        title="VEI vs. Impact",
        template="plotly_dark"
    )
    fig.update_traces(marker=dict(color='#ff5722', opacity=0.7))
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(color="white")
    )
    return fig

## UI for Dashboard

In [87]:
# Initialize App
app = Dash(__name__)

# Load Data
df = load_data()

# Styles
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#111111",
    "color": "white"
}

CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
    "background-color": "#000000",
    "min-height": "100vh",
    "color": "white"
}

CARD_STYLE = {
    "background-color": "#1e1e1e",
    "padding": "20px",
    "border-radius": "10px",
    "margin-bottom": "20px",
    "box-shadow": "0 4px 6px rgba(0,0,0,0.3)"
}

# Layout
app.layout = html.Div([
    # Sidebar
    html.Div([
        html.H2("Volcano Insights", style={'font-size': '20px', 'margin-bottom': '20px', 'color': '#ff5722'}),
        html.Hr(style={'border-color': '#333'}),
        html.P("Filters", style={'color': '#888'}),
        
        html.Label("Year Range", style={'margin-top': '20px'}),
        dcc.RangeSlider(
            id='year-slider',
            min=df['Year'].min(),
            max=df['Year'].max(),
            value=[df['Year'].min(), df['Year'].max()],
            marks={str(year): str(year) for year in range(int(df['Year'].min()), int(df['Year'].max()), 1000)},
            tooltip={"placement": "bottom", "always_visible": True},
            className="dark-slider"
        ),
        
        html.Label("Country", style={'margin-top': '20px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=[{'label': c, 'value': c} for c in sorted(df['Country'].unique())],
            placeholder="All Countries",
            style={'color': 'black'}  # dropdown text
        )
    ], style=SIDEBAR_STYLE),

    # Main Content
    html.Div([
        html.H1("Volcano Insights Dashboard", style={'margin-bottom': '5px'}),
        html.P("Analyzing Significant Volcanic Eruptions", style={'color': '#888', 'margin-bottom': '30px'}),

        # KPI Row
        html.Div([
            html.Div([
                html.H4("Total Eruptions", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-eruptions', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Deaths", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-deaths', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([
                html.H4("Total Damage ($M)", style={'color': '#888', 'font-size': '14px'}),
                html.H2(id='kpi-damage', style={'font-size': '32px'})
            ], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex', 'justify-content': 'space-between', 'margin-bottom': '20px'}),

        # Charts Row 1
        html.Div([
            html.Div([dcc.Graph(id='map-graph')], style={**CARD_STYLE, 'flex': '2', 'margin-right': '20px'}),
        ], style={'display': 'flex', 'margin-bottom': '20px'}),

        # Charts Row 2
        html.Div([
            html.Div([dcc.Graph(id='impact-graph')], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),
            html.Div([dcc.Graph(id='corr-graph')], style={**CARD_STYLE, 'flex': '1'})
        ], style={'display': 'flex'}),

        # Charts Row 3 - Temporal Analysis
        html.Div([
            html.H4(
                "Temporal dynamics",
                style={
                    'margin-bottom': '5px',
                    'color': 'white',
                    'font-size': '26px',
                    'font-weight': 'bold',
                    'font-family': '"Open Sans", "Helvetica", "Arial", sans-serif'
                }
            ),

            html.Div([
                # LEFT GRAPH (time series with mode selector)
                html.Div([
                    dcc.RadioItems(
                        id='ts-mode',
                        options=[
                            {'label': 'Per year', 'value': 'year'},
                            {'label': 'Per century', 'value': 'century'},
                            {'label': 'Smoothed (10-year MA)', 'value': 'smooth'},
                        ],
                        value='year',
                        inline=True,
                        style={'margin-bottom': '10px', 'color': 'white'}
                    ),
                    dcc.Graph(id='time-series-graph', style={'height': '330px'})
                ], style={**CARD_STYLE, 'flex': '1', 'margin-right': '20px'}),

                # RIGHT GRAPH (forecast)
                html.Div([
                    dcc.Graph(id='forecast-year-graph', style={'height': '330px'})
                ], style={**CARD_STYLE, 'flex': '1'}),
            ], style={'display': 'flex'})
        ])
    ], style=CONTENT_STYLE)
])


@app.callback(
    [
        Output('map-graph', 'figure'),
        Output('impact-graph', 'figure'),
        Output('corr-graph', 'figure'),
        Output('time-series-graph', 'figure'),
        Output('forecast-year-graph', 'figure'),
        Output('kpi-eruptions', 'children'),
        Output('kpi-deaths', 'children'),
        Output('kpi-damage', 'children'),
    ],
    [
        Input('country-dropdown', 'value'),
        Input('year-slider', 'value'),
        Input('ts-mode', 'value')
    ]
)
def update_dashboard(selected_country, year_range, ts_mode):
    # Filter Data
    dff = df.copy()
    if selected_country:
        dff = dff[dff['Country'] == selected_country]
    
    if year_range:
        dff = dff[(dff['Year'] >= year_range[0]) & (dff['Year'] <= year_range[1])]

    if dff.empty:
        dff = df  # fallback global

    # KPIs
    total_eruptions = len(dff)
    total_deaths = f"{int(dff['Deaths'].sum()):,}"
    total_damage = f"${dff['Damage_Millions'].sum():,.0f}"

    # Main figures
    fig_map = get_map_figure(dff)
    fig_impact = get_impact_figure(dff)
    fig_corr = get_correlation_figure(dff)

    # Temporal analysis: year / century / smooth
    fig_ts = render_temporal_series(dff, mode=ts_mode)

    # Forecast per year (modern period only, e.g. >= 1800)
    eruptions_per_year = build_eruptions_per_year(dff)
    df_prophet_year = prepare_prophet_df(eruptions_per_year, min_year=1800)

    if len(df_prophet_year) > 5:
        prediction_year = forecast_eruptions(df_prophet_year, n_future_years=20)
        fig_forecast_year = make_forecast_figure(df_prophet_year, prediction_year)
    else:
        fig_forecast_year = make_empty_forecast_figure("Not enough data for yearly forecast")

    return (
        fig_map,            # map-graph
        fig_impact,         # impact-graph
        fig_corr,           # corr-graph
        fig_ts,             # time-series-graph
        fig_forecast_year,  # forecast-year-graph
        total_eruptions,    # KPI
        total_deaths,       # KPI
        total_damage        # KPI
    )


if __name__ == '__main__':
    print("Launching Dashboard...")
    print("Dashboard launched at: http://127.0.0.1:7860")
    app.run(host='127.0.0.1', port=7860, debug=True)


Launching Dashboard...
Dashboard launched at: http://127.0.0.1:7860


20:17:35 - cmdstanpy - INFO - Chain [1] start processing
20:17:35 - cmdstanpy - INFO - Chain [1] done processing
20:17:40 - cmdstanpy - INFO - Chain [1] start processing
20:17:40 - cmdstanpy - INFO - Chain [1] done processing
